#Preprocessing

Your responsibility is to prepare **Member 4's approved feature dataset** so that machine-learning algorithms can use it correctly.

**Input from Member 4:** `data/processed/hotel_bookings_features.csv`

Member 4 has already completed feature engineering, feature selection, and feature-level leakage/timing decisions. This notebook does not redo those decisions.

You need to:
- Identify the approved numerical, binary, and categorical features for preprocessing.
- Decide how categorical variables should be encoded.
- Use One-Hot Encoding where appropriate.
- Investigate which numerical features need scaling.
- Use `StandardScaler` where appropriate.
- Prepare the stratified train/test split.
- Build the reusable preprocessing pipeline.
- Fit preprocessing on training data only.
- Save the shared preprocessing assets for Stage 2.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Setup and load Member 4's approved feature dataset

Member 4 has already:
- removed the approved leakage/timing-sensitive predictors,
- created `total_stay`,
- created `total_guests`,
- created `is_family`,
- saved the final Stage 1 feature dataset.

This notebook starts from that handover and performs preprocessing only.


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import sklearn

RANDOM_STATE = 42   # fixed seed -> everyone gets the same split

# Same folder structure as the previous team notebooks
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/hotel-booking-cancellation')
except ImportError:
    # Local Jupyter / VS Code: find the project folder automatically
    ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'data/processed/hotel_bookings_features.csv').exists()
    )

FEATURE_PATH = ROOT / 'data/processed/hotel_bookings_features.csv'
SPLIT_DIR = ROOT / 'data/splits'
MODEL_DIR = ROOT / 'models'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(FEATURE_PATH)

# Member 2 converted agent IDs to text; keep them categorical.
if 'agent' in df.columns:
    df['agent'] = df['agent'].astype(str)

print('scikit-learn:', sklearn.__version__, '| pandas:', pd.__version__)
print(f'Loaded Member 4 feature dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
display(df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
scikit-learn: 1.6.1 | pandas: 2.2.3
Loaded Member 4 feature dataset: 87,222 rows x 29 columns


,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,deposit_type,agent,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_stay,total_guests,is_family,is_canceled
0,Resort Hotel,342,2015,July,27,1,0,0,2,0.0,...,No Deposit,No_Agent,Transient,0.0,0,0,0,2.0,0,0
1,Resort Hotel,737,2015,July,27,1,0,0,2,0.0,...,No Deposit,No_Agent,Transient,0.0,0,0,0,2.0,0,0
2,Resort Hotel,7,2015,July,27,1,0,1,1,0.0,...,No Deposit,No_Agent,Transient,75.0,0,0,1,1.0,0,0
3,Resort Hotel,13,2015,July,27,1,0,1,1,0.0,...,No Deposit,304,Transient,75.0,0,0,1,1.0,0,0
4,Resort Hotel,14,2015,July,27,1,0,2,2,0.0,...,No Deposit,240,Transient,98.0,0,1,2,2.0,0,0


### 1.1 Target balance after cleaning
We check how many bookings are cancelled. This tells us whether we need **stratification** in the split.


In [3]:
target_table = df['is_canceled'].value_counts().rename('bookings').to_frame()
target_table['percentage'] = (target_table['bookings'] / len(df) * 100).round(2)
display(target_table)

,bookings,percentage
is_canceled,,
0,63213,72.47
1,24009,27.53


## 2. Define approved features and target

Member 4 has already decided **which predictors are allowed**.

Here we only separate the approved dataset into:
- predictor features (`X`)
- target (`y = is_canceled`)

No feature engineering or feature-level leakage decisions are made in this notebook.


In [4]:
TARGET = 'is_canceled'

X = df.drop(columns=[TARGET])
y = df[TARGET]
print('Predictor columns kept:', X.shape[1])

Predictor columns kept: 28


## 3. Identify numerical, binary, and categorical features

The columns below are **Member 4's approved features**. Member 3 now categorizes them only by how they must be preprocessed.

| Type | Meaning | Example |
|---|---|---|
| Numerical | Numbers where magnitude has meaning | `lead_time`, `adr`, `total_stay`, `total_guests` |
| Binary | Already 0/1 | `is_repeated_guest`, `is_family` |
| Categorical | Labels with no numerical order | `hotel`, `country`, `agent` |

This is a preprocessing classification, not a new feature-selection decision.


In [5]:
NUMERICAL_FEATURES = [
    'lead_time',
    'arrival_date_year',
    'arrival_date_week_number',
    'arrival_date_day_of_month',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'adr',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'total_stay',
    'total_guests',
]

BINARY_FEATURES = [
    'is_repeated_guest',
    'is_family',
]

CATEGORICAL_FEATURES = [
    'hotel',
    'arrival_date_month',
    'meal',
    'country',
    'market_segment',
    'distribution_channel',
    'reserved_room_type',
    'deposit_type',
    'customer_type',
    'agent',
]

# Safety check: every Member 4-approved predictor is assigned to exactly one preprocessing group.
assigned = NUMERICAL_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES

assert len(assigned) == len(set(assigned)), 'A column is listed twice'

missing_from_lists = set(X.columns) - set(assigned)
extra_in_lists = set(assigned) - set(X.columns)

assert not missing_from_lists, f'Approved features not assigned to a preprocessing group: {missing_from_lists}'
assert not extra_in_lists, f'Listed features not present in Member 4 dataset: {extra_in_lists}'

feature_types = pd.DataFrame({
    'group': (
        ['numerical'] * len(NUMERICAL_FEATURES)
        + ['binary'] * len(BINARY_FEATURES)
        + ['categorical'] * len(CATEGORICAL_FEATURES)
    ),
    'pandas dtype': [str(X[c].dtype) for c in assigned],
    'unique values': [X[c].nunique() for c in assigned],
}, index=assigned)

display(feature_types)


,group,pandas dtype,unique values
lead_time,numerical,int64,479
arrival_date_year,numerical,int64,3
arrival_date_week_number,numerical,int64,53
arrival_date_day_of_month,numerical,int64,31
stays_in_weekend_nights,numerical,int64,17
stays_in_week_nights,numerical,int64,33
adults,numerical,int64,14
children,numerical,float64,5
babies,numerical,int64,5
previous_cancellations,numerical,int64,15


## 4. How should categorical variables be encoded?
Models only understand numbers, so text labels must be converted.

**Options we considered**
- **Label/Ordinal encoding** (City=0, Resort=1, ...) - ❌ creates a fake order. Logistic Regression would think country 150 is "more" than country 3.
- **One-Hot Encoding** - ✅ makes one 0/1 column per category (e.g. `hotel_City Hotel`, `hotel_Resort Hotel`). No fake order. Works for all 4 models.

**Problem:** `country` and `agent` have hundreds of categories - many with only a handful of bookings.
One column per rare category = many almost-empty columns and categories the model can't learn from.

**Decision:** One-Hot Encoding with `min_frequency` - categories seen fewer than `MIN_CATEGORY_COUNT` times **in the training set** are grouped into one `infrequent` column.
`handle_unknown='infrequent_if_exist'` means a brand-new country appearing in the test set (or in future data) goes into that same `infrequent` column instead of crashing the model.

In [6]:
MIN_CATEGORY_COUNT = 50   # can be tuned with the team

cardinality = pd.DataFrame({
    'unique categories': X[CATEGORICAL_FEATURES].nunique(),
    f'categories with >= {MIN_CATEGORY_COUNT} rows': [
        (X[c].value_counts() >= MIN_CATEGORY_COUNT).sum() for c in CATEGORICAL_FEATURES],
}).sort_values('unique categories', ascending=False)
display(cardinality)

,unique categories,categories with >= 50 rows
agent,334,87
country,178,51
arrival_date_month,12,12
reserved_room_type,9,8
market_segment,8,7
distribution_channel,5,4
customer_type,4,4
meal,4,4
deposit_type,3,3
hotel,2,2


## 5. Which numerical features need scaling?
Numerical features are on very different ranges (e.g. `lead_time` up to ~700 days, `babies` 0–10, `adr` up to ~5,000).

- **Logistic Regression** is affected by this: large-range features dominate, and the solver converges slowly. ➜ needs scaling.
- **Decision Tree, Random Forest, XGBoost** split on thresholds (`lead_time > 60?`), so scaling doesn't change them — but it doesn't hurt them either.

**Decision:** apply `StandardScaler` to all numerical features (each becomes mean 0, std 1: `z = (x − mean) / std`).
Because we use ONE shared pipeline for all four models, this keeps things simple and fair.
`is_repeated_guest` is already 0/1, so it is passed through unchanged.

In [7]:
num_summary = X[NUMERICAL_FEATURES].describe().T[['mean', 'std', 'min', 'max']]
num_summary['skewness'] = X[NUMERICAL_FEATURES].skew()
display(num_summary.round(2))

,mean,std,min,max,skewness
lead_time,79.98,86.06,0.0,737.0,1.43
arrival_date_year,2016.21,0.69,2015.0,2017.0,-0.30
arrival_date_week_number,26.83,13.67,1.0,53.0,0.02
arrival_date_day_of_month,15.82,8.84,1.0,31.0,0.00
stays_in_weekend_nights,1.00,1.03,0.0,19.0,1.34
stays_in_week_nights,2.62,2.04,0.0,50.0,2.58
adults,1.88,0.62,0.0,55.0,20.40
children,0.14,0.46,0.0,10.0,3.46
babies,0.01,0.11,0.0,10.0,21.13
previous_cancellations,0.03,0.37,0.0,26.0,34.33


## 6. Stratified train/test split
- **80% train / 20% test.**
- `stratify=y` ➜ both sets keep the same cancellation ratio.
- `random_state=42` ➜ everyone in the team gets exactly the same split.
- The split happens **before** fitting any scaler/encoder. The test set is locked away until final model evaluation.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

split_check = pd.DataFrame({
    'rows': [len(y), len(y_train), len(y_test)],
    'cancelled %': [y.mean() * 100, y_train.mean() * 100, y_test.mean() * 100],
}, index=['full cleaned data', 'train (80%)', 'test (20%)']).round(2)
display(split_check)

# Check: do any test rows look exactly the same as a training row (after dropping excluded columns)?
train_keys = set(map(tuple, X_train.astype(str).values))
test_in_train = pd.Series([tuple(r) in train_keys for r in X_test.astype(str).values])
print(f'Test rows identical to a training row: {test_in_train.sum()} '
      f'({test_in_train.mean() * 100:.1f}% of test)')

,rows,cancelled %
full cleaned data,87222,27.53
train (80%),69777,27.53
test (20%),17445,27.53


Test rows identical to a training row: 721 (4.1% of test)


## 7. Build the preprocessing with ColumnTransformer + Pipeline
- A **Pipeline** chains steps in order (e.g. impute ➜ scale).
- A **ColumnTransformer** applies a different pipeline to different column groups, then joins the results.

`SimpleImputer` is included as a safety net: Member 2's cleaned data has no missing values, but future/new data might.
It is also learnt from the training data only.

In [9]:
def build_preprocessor():
    '''Returns a NEW, UNFITTED preprocessor. Every model must use this same function.'''
    numeric_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                 min_frequency=MIN_CATEGORY_COUNT,
                                 sparse_output=False)),
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_pipeline, NUMERICAL_FEATURES),
            ('bin', 'passthrough', BINARY_FEATURES),
            ('cat', categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        remainder='drop',               # any column not listed is dropped
        verbose_feature_names_out=True,  # output names like num__lead_time, cat__hotel_City Hotel
    )
    return preprocessor.set_output(transform='pandas')

preprocessor = build_preprocessor()
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['lead_time', 'arrival_date_year',
                                  'arrival_date_week_number',
                                  'arrival_date_day_of_month',
                                  'stays_in_weekend_nights',
                                  'stays_in_week_nights', 'adults', 'children',
                                  'babies', 'previous_cancellations',
                                  'previous_bookings_not_cancel...
                                 ['is_repeated_guest', 'is_family']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=50,
                                                                sparse_output=False))]),
                                 ['hotel', 'arrival_date_month', 'meal',
                                  'country', 'market_segment',
                                  'distribution_channel', 'reserved_room_type',
                                  'deposit_type', 'customer_type', 'agent'])])

## 8. Fit on TRAIN only, then transform
- `fit_transform(X_train)` ➜ learns means/stds/category lists **from train** and transforms train.
- `transform(X_test)` ➜ applies what was learnt from train to test. It learns **nothing** from test.

In [10]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print('Train before:', X_train.shape, '-> after:', X_train_prepared.shape)
print('Test  before:', X_test.shape, '-> after:', X_test_prepared.shape)
print('Missing values after preprocessing (train, test):',
      int(X_train_prepared.isna().sum().sum()), int(X_test_prepared.isna().sum().sum()))

feature_names = preprocessor.get_feature_names_out()
blocks = pd.Series([n.split('__')[0] for n in feature_names]).value_counts()
print('\nOutput columns by block:'); print(blocks.to_string())
display(X_train_prepared.head())

Train before: (69777, 28) -> after: (69777, 195)
Test  before: (17445, 28) -> after: (17445, 195)
Missing values after preprocessing (train, test): 0 0

Output columns by block:
cat    177
num     16
bin      2


,num__lead_time,num__arrival_date_year,num__arrival_date_week_number,num__arrival_date_day_of_month,num__stays_in_weekend_nights,num__stays_in_week_nights,num__adults,num__children,num__babies,num__previous_cancellations,...,cat__agent_85,cat__agent_86,cat__agent_89,cat__agent_9,cat__agent_94,cat__agent_95,cat__agent_96,cat__agent_98,cat__agent_No_Agent,cat__agent_infrequent_sklearn
68041,-0.916855,1.151596,0.596389,1.489521,-0.978129,-0.30534,-1.475128,-0.306151,-0.094677,-0.08025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
64474,-0.928483,-0.307095,0.011553,-1.563712,-0.978129,-1.28453,0.204786,-0.306151,-0.094677,-0.08025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14912,-0.730792,-1.765786,1.619852,-1.337546,0.968254,-0.30534,0.204786,-0.306151,8.518449,-0.08025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
15025,-0.463328,-1.765786,1.839166,0.811025,-0.978129,-0.30534,1.884700,-0.306151,-0.094677,2.54251,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26070,-0.905226,-0.307095,1.839166,0.811025,-0.978129,-1.28453,0.204786,-0.306151,-0.094677,-0.08025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 8.1 Proof there is no test-set leakage
If the scaler had seen the test set, its stored mean would equal the mean of the *full* data.
Instead it should exactly equal the **training** mean.

In [11]:
scaler = preprocessor.named_transformers_['num'].named_steps['scaler']
leak_check = pd.DataFrame({
    'scaler mean (stored)': scaler.mean_,
    'train mean': X_train[NUMERICAL_FEATURES].mean().values,
    'full-data mean': X[NUMERICAL_FEATURES].mean().values,
}, index=NUMERICAL_FEATURES).round(4)
display(leak_check.head(6))
assert np.allclose(scaler.mean_, X_train[NUMERICAL_FEATURES].mean().values)
print('✅ Scaler was learnt from the training set only.')

# Scaled train columns should now be mean ~0, std ~1
print('\nScaled train mean (should be ~0):', round(X_train_prepared.filter(like='num__').mean().abs().max(), 6))
print('Scaled train std  (should be ~1):', round(X_train_prepared.filter(like='num__').std().mean(), 3))

,scaler mean (stored),train mean,full-data mean
lead_time,79.8428,79.8428,79.9755
arrival_date_year,2016.2105,2016.2105,2016.2104
arrival_date_week_number,26.8420,26.8420,26.8347
arrival_date_day_of_month,15.8280,15.8280,15.8161
stays_in_weekend_nights,1.0051,1.0051,1.0046
stays_in_week_nights,2.6237,2.6237,2.6240


✅ Scaler was learnt from the training set only.

Scaled train mean (should be ~0): 0.0
Scaled train std  (should be ~1): 1.0


### 8.2 What happens to rare / unseen categories?

In [12]:
onehot = preprocessor.named_transformers_['cat'].named_steps['onehot']
rare_summary = pd.DataFrame({
    'categories in train': [X_train[c].nunique() for c in CATEGORICAL_FEATURES],
    'kept as own column': [len(cats) - (0 if inf is None else len(inf))
                           for cats, inf in zip(onehot.categories_, onehot.infrequent_categories_)],
    'grouped into infrequent': [0 if inf is None else len(inf) for inf in onehot.infrequent_categories_],
}, index=CATEGORICAL_FEATURES)
display(rare_summary)

unseen_countries = set(X_test['country']) - set(X_train['country'])
print('Countries in test but never seen in train:', len(unseen_countries), sorted(unseen_countries)[:10])
print('-> These are placed in country_infrequent_sklearn, so the model does not crash.')

,categories in train,kept as own column,grouped into infrequent
hotel,2,2,0
arrival_date_month,12,12,0
meal,4,4,0
country,169,47,122
market_segment,8,7,1
distribution_channel,5,4,1
reserved_room_type,9,8,1
deposit_type,3,3,0
customer_type,4,4,0
agent,316,81,235


Countries in test but never seen in train: 9 ['BFA', 'BRB', 'CYM', 'GGY', 'KIR', 'LCA', 'MLI', 'PYF', 'SLE']
-> These are placed in country_infrequent_sklearn, so the model does not crash.


## 9. Preprocessing Pipeline Completeness
Our unified preprocessing pipeline is designed to be fully compatible with any standard downstream machine-learning estimator.

In [13]:
# Verify the ColumnTransformer structure is correct and unfitted
preprocessor_check = build_preprocessor()
print('Preprocessor pipeline generated and verified.')
print('Output features mapped:', len(NUMERICAL_FEATURES) + len(BINARY_FEATURES) + len(CATEGORICAL_FEATURES), 'raw features.')

Preprocessor pipeline generated and verified.
Output features mapped: 28 raw features.


## 10. Save everything for the team

We save the **raw approved-feature split** plus the **unfitted preprocessing pipeline**.

Each Stage 2 model should use:

`Pipeline([('preprocess', preprocessor), ('model', ...)]).fit(X_train, y_train)`

Why not save only already-scaled/encoded data? Because preprocessing should be re-fitted correctly inside later cross-validation/tuning workflows rather than leaking validation information.


In [14]:
X_train.to_csv(SPLIT_DIR / 'X_train.csv', index=False)
X_test.to_csv(SPLIT_DIR / 'X_test.csv', index=False)
y_train.to_frame().to_csv(SPLIT_DIR / 'y_train.csv', index=False)
y_test.to_frame().to_csv(SPLIT_DIR / 'y_test.csv', index=False)

joblib.dump(build_preprocessor(), MODEL_DIR / 'preprocessor_unfitted.joblib')

config = {
    'source_feature_dataset': str(FEATURE_PATH),
    'target': TARGET,
    'random_state': RANDOM_STATE,
    'test_size': 0.20,
    'stratified': True,
    'numerical_features': NUMERICAL_FEATURES,
    'binary_features': BINARY_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'min_category_count': MIN_CATEGORY_COUNT,
    'train_rows': len(X_train),
    'test_rows': len(X_test),
    'sklearn_version': sklearn.__version__,
}

(MODEL_DIR / 'preprocessing_config.json').write_text(
    json.dumps(config, indent=2)
)

print('Saved to:')
for p in [
    SPLIT_DIR / 'X_train.csv',
    SPLIT_DIR / 'X_test.csv',
    SPLIT_DIR / 'y_train.csv',
    SPLIT_DIR / 'y_test.csv',
    MODEL_DIR / 'preprocessor_unfitted.joblib',
    MODEL_DIR / 'preprocessing_config.json'
]:
    print(' ', p)


Saved to:
  /content/drive/MyDrive/hotel-booking-cancellation/data/splits/X_train.csv
  /content/drive/MyDrive/hotel-booking-cancellation/data/splits/X_test.csv
  /content/drive/MyDrive/hotel-booking-cancellation/data/splits/y_train.csv
  /content/drive/MyDrive/hotel-booking-cancellation/data/splits/y_test.csv
  /content/drive/MyDrive/hotel-booking-cancellation/models/preprocessor_unfitted.joblib
  /content/drive/MyDrive/hotel-booking-cancellation/models/preprocessing_config.json


### 10.1 How teammates load preprocessing assets

In [15]:
from sklearn.base import clone

# Load the shared Member 4-approved split and preprocessing pipeline.
X_train_s = pd.read_csv(
    SPLIT_DIR / 'X_train.csv',
    dtype={'agent': str}
)
y_train_s = pd.read_csv(SPLIT_DIR / 'y_train.csv')['is_canceled']

shared_preprocessor = joblib.load(
    MODEL_DIR / 'preprocessor_unfitted.joblib'
)

# Clone a fresh copy of the preprocessing pipeline.
preprocessor_inst = clone(shared_preprocessor)

print('Unfitted preprocessor successfully loaded and cloned.')


Unfitted preprocessor successfully loaded and cloned.


## 11. Preprocessing decision summary

In [16]:
summary = pd.DataFrame([
    ('Input feature set', 'Member 4 approved feature dataset', 'Feature engineering/selection/leakage decisions are already complete'),
    ('Feature types', 'Assigned by preprocessing meaning', 'IDs like agent are labels, not quantities'),
    ('Categorical encoding', 'One-Hot Encoding', 'No fake order; compatible with planned models'),
    ('Rare categories', f'Grouped when < {MIN_CATEGORY_COUNT} train rows', 'country/agent contain many rare levels'),
    ('Unseen categories', "handle_unknown='infrequent_if_exist'", 'New countries/agents in test/future data do not crash'),
    ('Numerical scaling', 'StandardScaler', 'Provides a consistent numerical preprocessing pipeline'),
    ('Binary features', 'Passthrough', 'Already represented as 0/1'),
    ('Missing values', 'SimpleImputer inside pipeline', 'Safety net for new data; learnt from train only'),
    ('Train/test split', '80/20, stratified, random_state=42', 'Preserves target balance and is reproducible'),
    ('Preprocessing leakage prevention', 'Split first; fit preprocessing only on train', 'Test set never influences fitted preprocessing'),
    ('Reusability', 'build_preprocessor() + saved joblib', 'Shared preprocessing for Stage 2 models'),
], columns=['Item', 'Decision', 'Reason'])

display(summary)


,Item,Decision,Reason
0,Input feature set,Member 4 approved feature dataset,Feature engineering/selection/leakage decision...
1,Feature types,Assigned by preprocessing meaning,"IDs like agent are labels, not quantities"
2,Categorical encoding,One-Hot Encoding,No fake order; compatible with planned models
3,Rare categories,Grouped when < 50 train rows,country/agent contain many rare levels
4,Unseen categories,handle_unknown='infrequent_if_exist',New countries/agents in test/future data do no...
5,Numerical scaling,StandardScaler,Provides a consistent numerical preprocessing ...
6,Binary features,Passthrough,Already represented as 0/1
7,Missing values,SimpleImputer inside pipeline,Safety net for new data; learnt from train only
8,Train/test split,"80/20, stratified, random_state=42",Preserves target balance and is reproducible
9,Preprocessing leakage prevention,Split first; fit preprocessing only on train,Test set never influences fitted preprocessing


## 12. Handover notes

**Input received **
- `hotel_bookings_features.csv`
- leakage/timing-sensitive fields already removed
- engineered features already added: `total_stay`, `total_guests`, `is_family`

**To the modelling stage**
- Use the saved `X_train`, `X_test`, `y_train`, and `y_test`.
- Use the saved unfitted preprocessing pipeline with the model pipeline.
- Fit preprocessing/model workflows using training data.
- Keep `X_test` for final evaluation.

Member 3 does not redo Member 4's feature engineering, feature selection, or feature-level leakage decisions.
